# M9.7 · Köşe tespiti — Harris-Stephens, Shi-Tomasi, FAST

Kayan pencerenin kare fark toplamından Harris matrisine, özdeğerlerden $R=\det M-k\,\mathrm{tr}^2M$ ölçüsüne. Notlardaki vaka çalışmasının matrislerini elle ve saf Python'da birebir yeniden üretiyorsun; NumPy'da `cv2.cornerHarris` ve `cv2.cornerMinEigenVal`'ı ölçek çarpanlarıyla birlikte aynı sonuca getiriyorsun; PyTorch'ta Gauss ağırlıklı Harris ile dönme ve ölçek altında tekrarlanabilirliği ölçüp Harris'in neden dönmeye dayanıklı ama ölçeğe duyarlı olduğunu (ve SIFT'in neden gerektiğini) görüyorsun; en sonda yüzlerce köşeyi aynı anda alt piksel hassasiyetine taşıyan toplu bir iyileştirme yazıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-7-corners.html](https://m-gonul.github.io/pytorch-lab/m9-7-corners.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_7_corners.py
CHECKS = {
 "b": "import numpy as np\nfrom scipy import ndimage\nfrom skimage import data\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef ref_resp(img, k, r):\n    f = img.astype(np.float64)\n    H, W = f.shape\n    fx = np.zeros_like(f); fy = np.zeros_like(f)\n    fx[1:-1, 1:-1] = f[1:-1, 2:] - f[1:-1, :-2]; fy[1:-1, 1:-1] = f[2:, 1:-1] - f[:-2, 1:-1]\n    box = lambda a: ndimage.uniform_filter(a, 2 * r + 1, mode=\"constant\") * (2 * r + 1) ** 2\n    a, b, c = box(fx * fx), box(fx * fy), box(fy * fy)\n    R = a * c - b * b - k * (a + c) ** 2\n    out = np.zeros_like(R); out[r + 1:H - r - 1, r + 1:W - r - 1] = R[r + 1:H - r - 1, r + 1:W - r - 1]\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"cv2\", \"scipy\", \"skimage\"}))\n    G = 255 ** 2\n\n    def case():\n        left = [[0, 255, 255, 255, 255], [0, 0, 255, 255, 255], [0, 0, 0, 255, 255], [0, 0, 0, 0, 255], [0, 0, 0, 0, 0]]\n        right = [[255] * 5, [255] * 5, [0, 0, 0, 255, 255], [0, 0, 0, 255, 255], [0, 0, 0, 255, 255]]\n        ensure(need(ns, \"harris_matrix\")(left) == [[5 * G, -5 * G], [-5 * G, 5 * G]], f\"sol blok: {ns['harris_matrix'](left)}\")\n        ensure(ns[\"harris_matrix\"](right) == [[4 * G, -G], [-G, 4 * G]], f\"sağ blok: {ns['harris_matrix'](right)}\")\n    R.test(\"notlardaki vaka çalışması\", case)\n\n    img = data.camera()[200:240, 230:270].astype(int)\n\n    def resp():\n        for r in (1, 2):\n            got = np.array(need(ns, \"harris_response\")(img.tolist(), 0.04, r), float)\n            close(got, ref_resp(img, 0.04, r), rtol=1e-9, atol=1e-3, what=f\"r = {r}\")\n    R.test(\"harris_response\", resp)\n\n    def lm():\n        Rm = ref_resp(img, 0.04, 1)\n        thr = 0.01 * Rm.max()\n        got = need(ns, \"local_maxima\")(Rm.tolist(), thr, 2)\n        mx = ndimage.maximum_filter(Rm, 5, mode=\"constant\", cval=-np.inf)\n        ys, xs = np.nonzero((Rm > thr) & (Rm >= mx))\n        ensure([tuple(p) for p in got] == list(zip(xs.tolist(), ys.tolist())), \"yerel en büyükler farklı\")\n        return f\"{len(got)} köşe\"\n    R.test(\"local_maxima\", lm)\n",
 "c": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    img = data.camera()\n\n    for block, ksize in [(2, 3), (3, 3), (5, 5), (3, 1)]:\n        def t(block=block, ksize=ksize):\n            got = need(ns, \"corner_harris_cv\")(img, block, ksize, 0.04)\n            want = cv2.cornerHarris(img, block, ksize, 0.04)\n            close(got, want, atol=1e-6 * float(np.abs(want).max()), what=\"cornerHarris\")\n            got2 = need(ns, \"corner_min_eig_cv\")(img, block, ksize)\n            want2 = cv2.cornerMinEigenVal(img, block, ksize=ksize)\n            close(got2, want2, atol=1e-6 * float(np.abs(want2).max()), what=\"cornerMinEigenVal\")\n        R.test(f\"= cv2.cornerHarris / cornerMinEigenVal (block {block}, ksize {ksize})\", t)\n\n    def pts():\n        rows = []\n        for k, T in [(0.04, 0.01), (0.1, 0.01), (0.04, 0.1), (0.2, 0.01)]:\n            p = need(ns, \"harris_points\")(img, k, T)\n            ensure(p.ndim == 2 and p.shape[1] == 2, \"(N, 2)\")\n            Rm = cv2.cornerHarris(img, 3, 3, k)\n            vals = Rm[p[:, 1], p[:, 0]]\n            ensure(np.all(np.diff(vals) <= 0), \"yanıta göre azalan sırada değil\")\n            rows.append([k, T, len(p)])\n        ensure(rows[1][2] < rows[0][2] and rows[2][2] < rows[0][2], \"k ya da T büyüyünce köşe sayısı azalmalı\")\n        R.table(\"camera, 7 × 7 yerel en büyük\", [\"k\", \"T (max'ın oranı)\", \"köşe\"], rows)\n    R.test(\"harris_points\", pts)\n",
 "d": "import math\nimport numpy as np\nimport torch\nfrom scipy import ndimage\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef ref_harris(img, sd, si, k):\n    def g(s):\n        r = int(math.ceil(3 * s)); t = np.arange(-r, r + 1); w = np.exp(-t ** 2 / (2 * s * s)); return w / w.sum()\n    sep = lambda a, w: ndimage.correlate1d(ndimage.correlate1d(a, w, 1, mode=\"mirror\"), w, 0, mode=\"mirror\")\n    x = sep(img, g(sd))\n    gx = ndimage.correlate1d(x, [-0.5, 0, 0.5], 1, mode=\"nearest\"); gy = ndimage.correlate1d(x, [-0.5, 0, 0.5], 0, mode=\"nearest\")\n    w = g(si)\n    a, b, c = sep(gx * gx, w), sep(gx * gy, w), sep(gy * gy, w)\n    return a * c - b * b - k * (a + c) ** 2\n\n\ndef check(ns, R):\n    img = torch.tensor(data.camera(), dtype=torch.float32) / 255\n\n    def h():\n        crop = img[100:200, 150:280]\n        for sd, si, k in [(1.0, 2.0, 0.04), (1.5, 3.0, 0.1)]:\n            got = need(ns, \"harris_torch\")(crop, sd, si, k).double().numpy()\n            want = ref_harris(crop.double().numpy(), sd, si, k)\n            close(got, want, atol=1e-5 * np.abs(want).max(), what=f\"σ_d = {sd}, σ_i = {si}\")\n    R.test(\"harris_torch\", h)\n\n    def det():\n        Rm = need(ns, \"harris_torch\")(img)\n        p = need(ns, \"detect\")(Rm, 50, 4, 16)\n        ensure(tuple(p.shape) == (50, 2), \"(n, 2)\")\n        x, y = p[:, 0].long(), p[:, 1].long()\n        ensure(bool(((x >= 16) & (x < 512 - 16) & (y >= 16) & (y < 512 - 16)).all()), \"kenar bölgesinde tespit var\")\n        vals = Rm[y, x]\n        mx = torch.nn.functional.max_pool2d(Rm[None, None], 9, 1, 4)[0, 0]\n        ensure(bool((vals == mx[y, x]).all()), \"yerel en büyük değil\")\n        ensure(bool((vals[:-1] >= vals[1:]).all()), \"azalan sırada değil\")\n    R.test(\"detect\", det)\n\n    def rep():\n        f = need(ns, \"repeatability\")\n        rows = [[f\"{a}°\", s, f\"{f(img, a, s):.2f}\"] for a, s in [(0, 1.0), (15, 1.0), (30, 1.0), (45, 1.0), (0, 1.5), (0, 2.0)]]\n        r30, rs2 = float(rows[2][2]), float(rows[5][2])\n        ensure(float(rows[0][2]) > 0.99, \"dönüşüm yokken tekrarlanabilirlik 1 olmalı\")\n        ensure(r30 > 0.75, f\"30° dönmede {r30:.2f}: Harris dönmeye dayanıklı olmalı\")\n        ensure(rs2 < r30 - 0.15, f\"2× ölçekte {rs2:.2f}, dönmede {r30:.2f}: ölçek daha çok zarar vermeli\")\n        fixed3, ad3 = f(img, 0, 3.0, False, 6.0), f(img, 0, 3.0, True, 6.0)\n        ensure(ad3 > fixed3 + 0.05, f\"3× ölçekte uyarlamalı {ad3:.2f}, sabit {fixed3:.2f}\")\n        rows.append([\"0°\", \"3.0 (tol 6)\", f\"sabit σ {fixed3:.2f} / uyarlamalı σ {ad3:.2f}\"])\n        R.table(\"camera, 200 köşe, 2 piksel tolerans\", [\"dönme\", \"ölçek\", \"tekrarlanabilirlik\"], rows,\n                note=\"Ölçeği bilinmeyen bir eşleştirmede σ'yı nasıl seçeceğiz? Cevap ölçek uzayı: M9.14'te SIFT.\")\n    R.test(\"dönme ve ölçek altında tekrarlanabilirlik\", rep)\n",
 "e": "import numpy as np\nimport cv2\nimport torch\nfrom labcheck import max_loops, need, ensure, bench, fmt_time\n\n\ndef scene(seed, n=6, S=48, up=8):\n    rng = np.random.default_rng(seed)\n    H = W = S * n\n    hi = np.full((H * up, W * up), 40, np.uint8)\n    pts = []\n    for i in range(n):\n        for j in range(n):\n            c = np.array([j * S + S / 2, i * S + S / 2]); k = rng.integers(3, 5)\n            ang = np.linspace(0, 2 * np.pi, k, endpoint=False) + rng.uniform(0, 2 * np.pi)\n            P = c + rng.uniform(13, 17) * np.stack([np.cos(ang), np.sin(ang)], 1)\n            X = (P + 0.5) * up - 0.5\n            cv2.fillPoly(hi, [np.round(X * 16).astype(np.int32)], int(rng.integers(150, 230)), lineType=cv2.LINE_8, shift=4)\n            pts.append(P)\n    return cv2.resize(hi, (W, H), interpolation=cv2.INTER_AREA), np.concatenate(pts)\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız iterasyon döngüsü\", lambda: max_loops(src, 1, func=\"refine_subpix\"))\n\n    def acc():\n        rows = []\n        for seed in (0, 1):\n            img, P = scene(seed)\n            init = np.round(P + np.random.default_rng(seed + 10).uniform(-1, 1, P.shape))\n            got = np.asarray(need(ns, \"refine_subpix\")(img, init), np.float64)\n            ensure(got.shape == P.shape, \"(N, 2)\")\n            e0, e1 = np.linalg.norm(init - P, axis=1).mean(), np.linalg.norm(got - P, axis=1).mean()\n            c = init.astype(np.float32).reshape(-1, 1, 2).copy()\n            cv2.cornerSubPix(img, c, (5, 5), (-1, -1), (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 0.001))\n            ec = np.linalg.norm(c[:, 0] - P, axis=1).mean()\n            ensure(e1 < 0.25 and e1 < e0 / 3, f\"ortalama hata {e1:.3f} piksel (başlangıç {e0:.3f})\")\n            ensure(e1 < ec + 0.05, f\"senin {e1:.3f}, cv2.cornerSubPix {ec:.3f}\")\n            rows.append([seed, len(P), f\"{e0:.3f}\", f\"{e1:.3f}\", f\"{ec:.3f}\"])\n        R.table(\"Alt piksel köşe hatası (piksel)\", [\"sahne\", \"köşe\", \"tamsayı başlangıç\", \"senin\", \"cv2.cornerSubPix\"], rows,\n                note=\"Kalan ~0.15 piksel, 8× üst örneklemeyle çizilmiş dar açılı köşelerin bilineer modelden sapmasından gelir.\")\n    R.test(\"alt piksel doğruluğu\", acc)\n    if \"refine_subpix\" not in ns:\n        return\n    img, P = scene(3, n=12)\n    init = np.round(P)\n    t1 = bench(lambda: ns[\"refine_subpix\"](img, init), repeat=3)\n    c = init.astype(np.float32).reshape(-1, 1, 2)\n    t2 = bench(lambda: cv2.cornerSubPix(img, c.copy(), (5, 5), (-1, -1), (cv2.TERM_CRITERIA_MAX_ITER, 10, 0)), repeat=3)\n    R.table(f\"{len(P)} köşe, 10 iterasyon\", [\"yöntem\", \"süre\"], [[\"toplu (grid_sample + solve)\", fmt_time(t1)], [\"cv2.cornerSubPix\", fmt_time(t2)]])\n",
 "x1": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    img = data.checkerboard() / 255.0\n\n    def eig():\n        l1, l2 = need(ns, \"eigen_maps\")(img)\n        gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3) / 8; gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3) / 8\n        b = lambda a: cv2.GaussianBlur(a, (0, 0), 1.5)\n        a, bb, c = b(gx * gx), b(gx * gy), b(gy * gy)\n        ensure(np.all(l1 >= l2 - 1e-12), \"λ1 ≥ λ2\")\n        close(l1 + l2, a + c, atol=1e-10, what=\"iz\")\n        close(l1 * l2, a * c - bb * bb, atol=1e-10, what=\"determinant\")\n        close(need(ns, \"harris_from_eigs\")(l1, l2, 0.04), a * c - bb * bb - 0.04 * (a + c) ** 2, atol=1e-10, what=\"R\")\n        flat = (l1 < 1e-4).mean(); edge = ((l1 > 1e-3) & (l2 < 1e-4)).mean(); corner = (l2 > 1e-3).mean()\n        return f\"düz %{flat * 100:.0f}, kenar %{edge * 100:.0f}, köşe %{corner * 100:.1f}\"\n    R.test(\"özdeğerler, iz ve determinant\", eig)\n\n    def sweep():\n        rows = []\n        counts = {}\n        for var in (0.006, 0.01):\n            noisy = img + np.random.default_rng(0).normal(0, np.sqrt(var), img.shape)\n            l1, l2 = ns[\"eigen_maps\"](noisy)\n            for k in (0.04, 0.1, 0.249):\n                Rm = ns[\"harris_from_eigs\"](l1, l2, k)\n                for T in (1e-4, 1e-3):\n                    counts[var, k, T] = need(ns, \"count_corners\")(Rm, T)\n                rows.append([var, k, counts[var, k, 1e-4], counts[var, k, 1e-3]])\n        for var in (0.006, 0.01):\n            ensure(counts[var, 0.04, 1e-4] >= counts[var, 0.1, 1e-4] >= counts[var, 0.249, 1e-4], \"k büyüdükçe köşe sayısı azalmalı\")\n            ensure(counts[var, 0.04, 1e-4] >= counts[var, 0.04, 1e-3], \"T büyüdükçe köşe sayısı azalmalı\")\n        R.table(\"Gürültülü damalı tahta (gerçek iç köşe: 49)\", [\"gürültü σ²\", \"k\", \"T = 1e-4\", \"T = 1e-3\"], rows,\n                note=\"Düşük T ve k'da gürültü sahte köşeler üretir; k → 0.25'te gerçek köşeler bile kaybolmaya başlar.\")\n    R.test(\"k ve T taraması\", sweep)\n",
 "x2": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    rows = []\n    for name in (\"camera\", \"coins\", \"astronaut\"):\n        def t(name=name):\n            im = getattr(data, name)()\n            if im.ndim == 3:\n                im = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY)\n            for th in (10, 25, 40):\n                kp = cv2.FastFeatureDetector_create(th, False).detect(im)\n                want = {(int(k.pt[0]), int(k.pt[1])) for k in kp}\n                m = need(ns, \"fast\")(im, th)\n                ys, xs = np.nonzero(m)\n                got = set(zip(xs.tolist(), ys.tolist()))\n                ensure(got == want, f\"t = {th}: {len(got ^ want)} nokta farklı\")\n                rows.append([name, th, len(want)])\n        R.test(f\"fast = cv2.FAST ({name})\", t)\n    if rows:\n        R.table(\"FAST-9 (nonmax kapalı)\", [\"görüntü\", \"t\", \"köşe\"], rows)\n    if \"fast\" in ns:\n        im = data.camera()\n        t1 = bench(lambda: ns[\"fast\"](im, 20), repeat=3)\n        t2 = bench(lambda: cv2.FastFeatureDetector_create(20, False).detect(im), repeat=3)\n        t3 = bench(lambda: cv2.cornerHarris(im, 3, 3, 0.04), repeat=3)\n        R.table(\"512 × 512\", [\"yöntem\", \"süre\"], [[\"senin FAST (NumPy)\", fmt_time(t1)], [\"cv2 FAST\", fmt_time(t2)], [\"cv2.cornerHarris\", fmt_time(t3)]])\n",
 "x3": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    rows = []\n    for name in (\"camera\", \"coins\", \"astronaut\"):\n        def t(name=name):\n            im = getattr(data, name)()\n            if im.ndim == 3:\n                im = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY)\n            for mc, q, md in [(100, 0.01, 10), (500, 0.05, 5), (50, 0.1, 20)]:\n                want = cv2.goodFeaturesToTrack(im, mc, q, md)[:, 0]\n                got = need(ns, \"good_features\")(im, mc, q, md)\n                ensure(got.shape == want.shape, f\"({mc}, {q}, {md}): {len(got)} köşe, {len(want)} bekleniyordu\")\n                ensure(np.array_equal(got, want), f\"({mc}, {q}, {md}): {int((got != want).any(1).sum())} köşe farklı\")\n                rows.append([name, mc, q, md, len(want)])\n        R.test(f\"good_features = cv2.goodFeaturesToTrack ({name})\", t)\n    if rows:\n        R.table(\"Birebir eşleşen Shi-Tomasi seçimleri\", [\"görüntü\", \"max\", \"kalite\", \"min uzaklık\", \"köşe\"], rows)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_7_corners as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Fikir:** küçük bir pencereyi her yöne kaydır. Düz bölgede hiçbir yönde, kenarda tek yönde, **köşede her yönde** değişim olur.

**Türetme:** ağırlık $w$ ile kare fark toplamı $C(x,y)=\sum_{s,t}w(s,t)\,[f(s+x,t+y)-f(s,t)]^2$. Birinci dereceden Taylor açılımı $f(s+x,t+y)\approx f+xf_x+yf_y$ ile

$$C(x,y)\approx\begin{bmatrix}x&y\end{bmatrix}M\begin{bmatrix}x\\y\end{bmatrix},\qquad M=\sum_{s,t}w(s,t)\begin{bmatrix}f_x^2&f_xf_y\\f_xf_y&f_y^2\end{bmatrix}.$$

$M$ (Harris matrisi) türevlerin yerel ağırlıklı kovaryansıdır: özvektörleri en büyük yayılım yönleri, özdeğerleri o yönlerdeki yayılım. **İki küçük** özdeğer → düz, **biri büyük** → kenar, **ikisi büyük** → köşe (ne yazık ki tek parlak noktalar da).

**Özdeğer hesaplamadan:** $\det M=\lambda_1\lambda_2$, $\mathrm{tr}M=\lambda_1+\lambda_2$,
$$R=\det M-k\,\mathrm{tr}^2M:$$
köşede büyük pozitif, kenarda büyük negatif, düzde küçük. $0<k<0.25$; küçük $k$ daha hassas (daha çok köşe). Köşe: $R>T$ (ve yerel en büyük).

**Ağırlık:** kutu (hızlı, gürültü azsa) ya da Gauss $w=e^{-(s^2+t^2)/2\sigma^2}$ (yumuşatma önemliyse; ayrıca dönmeye karşı izotropik).

**Shi-Tomasi:** $R=\min(\lambda_1,\lambda_2)$, kapalı formda $\frac{a+c}2-\sqrt{(\frac{a-c}2)^2+b^2}$.

**FAST:** pikselin etrafındaki 16 piksellik çemberde, $I_p+t$'den parlak ya da $I_p-t$'den koyu **ardışık 9** piksel varsa köşe. Türev yok, çok hızlı.

## 1 · Elle

<!-- çözüm -->

Sağ blok: $\det=16G^2-G^2=15G^2$, $\mathrm{tr}=8G$, $R=15G^2-0.04\cdot64G^2=12.44G^2$. Sol blok: $\det=0$, $R=-0.04\cdot100G^2=-4G^2$ (kenar).

İdeal köşe $\lambda_1=\lambda_2=\lambda$: $R=\lambda^2(1-4k)$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Vaka çalışması ($G=255^2$): sol blok $M=\begin{bmatrix}5G&-5G\\-5G&5G\end{bmatrix}$**

- `l1`: $\lambda_1/G$ (büyük)
- `l2`: $\lambda_2/G$
- `rl`: $R/G^2$, $k=0.04$

**Sağ blok $M=\begin{bmatrix}4G&-G\\-G&4G\end{bmatrix}$**

- `r1`: $\lambda_1/G$
- `r2`: $\lambda_2/G$
- `rr`: $R/G^2$, $k=0.04$

**$k$'nin sınırı ve işaret**

- `kmax`: $\lambda_1=\lambda_2$ olan ideal köşede $R>0$ kalması için $k$'nin üst sınırı
- `redge`: $\lambda=(50,\;0.1)$, $k=0.04$ için $R$
- `npix`: Sol bloğun 3 × 3 iç bölgesinde $[-1,0,1]$ gradyanı sıfır olmayan piksel sayısı

In [ ]:
cevaplar = dict(
    l1=None,
    l2=None,
    rl=None,
    r1=None,
    r2=None,
    rr=None,
    kmax=None,
    redge=None,
    npix=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"l1":[10,0],"l2":[0,0],"rl":[-4,0],"r1":[5,0],"r2":[3,0],"rr":[12.44,0.001],"kmax":[0.25,0],"redge":[-95.4004,0.0001],"npix":[5,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, liste listeleri. Gradyanlar **merkezi fark** (notlardaki $w_y=[-1,0,1]$, $w_x=w_y^T$): $f_x=f(y,x+1)-f(y,x-1)$, $f_y=f(y+1,x)-f(y-1,x)$.

- `harris_matrix(patch)` — $n\times n$ yamanın **iç** pikselleri ($1..n-2$) üzerinde kutu ağırlıklı $M$ → `[[a, b], [b, c]]`. Notlardaki iki 5 × 5 blokla aynı matrisleri vermeli.
- `harris_response(img, k, r=1)` — her piksel için $(2r+1)^2$ kutu pencerede $M$ ve $R$; gradyan ya da pencere görüntü dışına taşan piksellerde (kenardan $r+1$ içeride olmayanlar) 0.
- `local_maxima(R, thr, r)` — `R > thr` ve $(2r+1)^2$ komşuluğundaki (görüntü içinde kalan) her değerden büyük ya da eşit olan noktalar, `(x, y)` listesi, satır sırasıyla.

In [ ]:
%%writefile stage_b.py
def harris_matrix(patch):
    ...


def harris_response(img, k, r=1):
    ...


def local_maxima(R, thr, r):
    ...


left = [[0, 255, 255, 255, 255], [0, 0, 255, 255, 255], [0, 0, 0, 255, 255], [0, 0, 0, 0, 255], [0, 0, 0, 0, 0]]
print(harris_matrix(left))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**; `cv2.Sobel`, `cv2.boxFilter` serbest.

`cv2.cornerHarris(img, block, ksize, k)` iç yapısı (`corner.cpp`):

1. Ölçek $s=\dfrac{1}{2^{\,ksize-1}\cdot block\cdot255}$ (uint8 girdi için; `ksize=1` iken $2^0$).
2. `dx = cv2.Sobel(f32, cv2.CV_32F, 1, 0, ksize=ksize, scale=s)`, `dy` benzer (varsayılan kenar kuralı).
3. $a,b,c$ = `cv2.boxFilter(dx*dx | dx*dy | dy*dy, -1, (block, block), normalize=False)`.
4. Harris: $ac-b^2-k(a+c)^2$. MinEigenVal: $a'=a/2$, $c'=c/2$, $(a'+c')-\sqrt{(a'-c')^2+b^2}$.

Yaz:

- `corner_harris_cv(img, block, ksize, k)` ve `corner_min_eig_cv(img, block, ksize)` — `float32`, OpenCV ile aynı.
- `harris_points(img, k=0.04, T_frac=0.01, r=3)` — `corner_harris_cv(img, 3, 3, k)` üzerinde `R > T_frac·max(R)` ve $(2r+1)^2$ yerel en büyük (`scipy.ndimage.maximum_filter`, `mode="constant", cval=-inf`); `(N, 2)` `[x, y]` dizisi, yanıta göre azalan.

In [ ]:
%%writefile stage_c.py
import numpy as np
import cv2
from scipy import ndimage


def corner_harris_cv(img, block, ksize, k):
    ...


def corner_min_eig_cv(img, block, ksize):
    ...


def harris_points(img, k=0.04, T_frac=0.01, r=3):
    ...


from skimage import data
print(harris_points(data.camera())[:5])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Gauss ağırlıklı Harris ve **tekrarlanabilirlik** (repeatability): görüntü bilinen bir dönüşümle değişince aynı köşelerin yeniden bulunma oranı.

- `harris_torch(img, sigma_d=1.0, sigma_i=2.0, k=0.04)` — `img (H, W)` float: (1) $\sigma_d$ Gauss ile yumuşat (ayrılabilir, yarıçap $\lceil3\sigma\rceil$, toplamı 1, `reflect` dolgu), (2) merkezi fark / 2 (`replicate`), (3) $f_x^2, f_xf_y, f_y^2$'yi $\sigma_i$ Gauss ile topla, (4) $R$.
- `detect(R, n=200, r=4, border=16)` — `max_pool2d` ile $(2r+1)^2$ yerel en büyükler, kenardan `border` piksel uzaktakiler arasından en büyük `n` tanesi; `(n, 2)` `[x, y]` float.
- `repeatability(img, angle, scale, adapt=False, tol=2.0)` — hazır `similarity` ile ikinci görüntüyü üret; ilkinde `harris_torch` + `detect`, ikincide `adapt` ise $\sigma_d\cdot s$, $\sigma_i\cdot s$ ve `r = int(4·s)`; ilk noktaları dönüşümle taşı, taşınanlardan ikinci görüntünün kenar bölgesine (16 piksel) düşmeyenler için en yakın tespite uzaklık `≤ tol` olanların oranı.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn.functional as F


def similarity(img, angle, scale):
    """Merkez etrafında döndür + ölçekle. (img2, A) döndürür; A (2, 3): p2 = A @ [x, y, 1]."""
    H, W = img.shape
    c = torch.tensor([W / 2, H / 2])
    th = math.radians(angle)
    Rm = torch.tensor([[math.cos(th), -math.sin(th)], [math.sin(th), math.cos(th)]]) * scale
    inv = torch.linalg.inv(Rm)
    ys, xs = torch.meshgrid(torch.arange(H, dtype=torch.float32), torch.arange(W, dtype=torch.float32), indexing="ij")
    src = (torch.stack([xs, ys], -1) - c) @ inv.T + c
    grid = torch.stack([src[..., 0] / (W - 1) * 2 - 1, src[..., 1] / (H - 1) * 2 - 1], -1)[None]
    img2 = F.grid_sample(img[None, None], grid, align_corners=True)[0, 0]
    return img2, torch.cat([Rm, (c - Rm @ c)[:, None]], 1)


def harris_torch(img, sigma_d=1.0, sigma_i=2.0, k=0.04):
    ...


def detect(R, n=200, r=4, border=16):
    ...


def repeatability(img, angle, scale, adapt=False, tol=2.0):
    ...


from skimage import data
img = torch.tensor(data.camera(), dtype=torch.float32) / 255
print(repeatability(img, 30, 1.0))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`refine_subpix(img, q, win=5, iters=10)` — `img (H, W)` uint8 dizisi, `q (N, 2)` başlangıç köşeleri `[x, y]`; **tüm köşeler aynı anda**:

1. Her köşe için $(2\,win+3)^2$ noktalık ızgarayı `grid_sample` (bilineer, `align_corners=True`) ile örnekle.
2. İç $(2\,win+1)^2$ noktada merkezi fark / 2 ile $g=(g_x,g_y)$; ağırlık $w=e^{-(dx^2+dy^2)/(2(win/2)^2)}$.
3. $A=\sum w\,gg^T$, $b=\sum w\,gg^Tp$ ($p$ örnek noktasının koordinatı); $q\leftarrow A^{-1}b$ (`torch.linalg.solve` ile $N$ sistem birden).

Fikir: gerçek köşede, köşeden pencere noktasına giden vektör ya bir kenar üzerindedir ($g\approx0$ değilse ona dik) ya da düz bölgededir ($g=0$); her iki durumda $g^T(p-q)=0$. Yalnız `iters` döngüsü serbest.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def refine_subpix(img, q, win=5, iters=10):
    """img (H, W) uint8 dizi, q (N, 2) [x, y]. (N, 2) tensör döndür."""
    ...


import numpy as np
sq = np.zeros((32, 32), np.uint8); sq[8:24, 8:24] = 200
print(refine_subpix(sq, np.array([[9.0, 7.0]])))  # köşe ≈ (7.5, 7.5)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Özdeğer haritaları, R ve k–T taraması

- `eigen_maps(img, sigma=1.5)` — `img` [0,1] float64; `cv2.Sobel(…, ksize=3) / 8` gradyanları, `cv2.GaussianBlur(…, (0, 0), sigma)` ile $a,b,c$; kapalı formdan $\lambda_1\ge\lambda_2$.
- `harris_from_eigs(l1, l2, k)` — $\lambda_1\lambda_2-k(\lambda_1+\lambda_2)^2$ (aynı $a,b,c$'den hesaplanan $ac-b^2-k(a+c)^2$ ile eşit olmalı).
- `count_corners(R, T)` — `R > T` olan 3 × 3 yerel en büyüklerin sayısı.

Kontrol, gürültülü damalı tahtada $k\in\{0.04,0.1,0.249\}$ ve $T$ ile köşe sayısının tablosunu çıkarır (notlardaki taramalar).

In [ ]:
%%writefile extra_x1.py
import numpy as np
import cv2


def eigen_maps(img, sigma=1.5):
    ...


def harris_from_eigs(l1, l2, k):
    ...


def count_corners(R, T):
    ...


from skimage import data
l1, l2 = eigen_maps(data.checkerboard() / 255.0)
print(l1.max(), l2.max())

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · FAST: 16 piksellik çember, 9'luk yay (cv2 ile birebir)

`fast(img, t)` — `bool` maske, `cv2.FastFeatureDetector_create(t, nonmaxSuppression=False)` ile **aynı** nokta kümesi. Çember (dx, dy) sırasıyla: `(0,3),(1,3),(2,2),(3,1),(3,0),(3,-1),(2,-2),(1,-3),(0,-3),(-1,-3),(-2,-2),(-3,-1),(-3,0),(-3,1),(-2,2),(-1,3)`. Parlak: $I_x-I_p>t$, koyu: $I_x-I_p<-t$; dairesel 16 elemanda ardışık 9 (ipucu: ilk 8'i sona ekle, kümülatif toplamla 9'luk pencere). Kenardan 3 piksel içeridekiler aday değil.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def fast(img, t):
    ...


from skimage import data
print(fast(data.camera(), 20).sum())

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Shi-Tomasi: cv2.goodFeaturesToTrack'i birebir yeniden yaz

`good_features(img, max_corners, quality, min_dist)` — `cv2.goodFeaturesToTrack(img, max_corners, quality, min_dist)[:, 0]` ile **aynı dizi** (sıra dahil). Algoritma (`featureselect.cpp`, varsayılan blok 3, Sobel 3, Shi-Tomasi):

1. `e = corner_min_eig` (c'deki gibi, float32).
2. `e > quality·max(e)` olmayanlar 0.
3. Aday: `e != 0` ve `e == cv2.dilate(e, None)`, kenardaki 1 piksel hariç.
4. Yanıta göre azalan sırada (kararlı sıralama) açgözlü seç: kabul edilmişlerin hepsine uzaklığın karesi $\ge$ `min_dist²` ise kabul; `max_corners`'a ulaşınca dur.

Çıkış `float32` `(N, 2)` `[x, y]`.

In [ ]:
%%writefile extra_x3.py
import numpy as np
import cv2


def good_features(img, max_corners, quality, min_dist):
    ...


from skimage import data
print(good_features(data.camera(), 10, 0.01, 10))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: SuperPoint: kendinden denetimli köşe ve betimleyici

[DeTone, Malisiewicz, Rabinovich — SuperPoint: Self-Supervised Interest Point Detection and Description (CVPRW 2018)](https://arxiv.org/abs/1712.07629)

Önce sentetik şekiller (üçgen, dörtgen, yıldız, damalı tahta, çizgiler) üzerinde köşe konumları bilinerek MagicPoint dedektörünü eğit; sonra homografik uyarlama (homographic adaptation) ile gerçek görüntülerde sözde etiket üretip SuperPoint'i (paylaşılan kodlayıcı, 65 kanallı "cell" çıkışı, betimleyici başı) eğit. HPatches'te Harris, Shi-Tomasi ve FAST ile tekrarlanabilirliği karşılaştır.


**Adımlar**

1. Sentetik şekil üretici ve köşe ısı haritası; 8 × 8 hücre başına 64 + 1 ('köşe yok') sınıflı softmax çıkışı.
2. Gürültü, bulanıklık, parlaklık artırmaları; Harris'in gürültüye karşı nasıl çöktüğünü ölç.
3. Homografik uyarlama: rastgele homografilerle çarpıtılmış kopyalarda tespit, geri çarpıtıp ortalama.
4. Betimleyici kaybı (hinge) ve eşleştirme ile homografi tahmini doğruluğu.

**Kabul ölçütleri**

- Sentetik test kümesinde σ = 0.2 gürültüde MagicPoint F1 > Harris F1 + 0.15.
- HPatches tekrarlanabilirlik tablosu (aydınlatma ve bakış açısı alt kümeleri ayrı).
- Homografik uyarlama öncesi ve sonrası sözde etiketlerin görsel karşılaştırması.

## Kaynaklar

- [Harris & Stephens — A Combined Corner and Edge Detector (Alvey 1988)](https://doi.org/10.5244/C.2.23) — ©
- [Shi & Tomasi — Good Features to Track (CVPR 1994)](https://doi.org/10.1109/CVPR.1994.323794) — ©
- [Rosten & Drummond — Machine learning for high-speed corner detection (ECCV 2006)](https://doi.org/10.1007/11744023_34) — ©
- [OpenCV — imgproc/src/corner.cpp, featureselect.cpp, features2d/src/fast.cpp](https://github.com/opencv/opencv/tree/4.x/modules) — Apache-2.0
- [Gonzalez & Woods — Digital Image Processing, 4. baskı, Bölüm 11.5](https://www.imageprocessingplace.com/) — ©